# Домашняя работа: квантизация и дистилляция LLM

В этой работе вы самостоятельно проведёте эксперименты с квантизацией и дистилляцией LLM и проверите, сколько качества при этом теряет модель. Задача, на которую обучаемся, — разбор запросов к голосовому ассистенту из датасета MASSIVE (ru-RU): по запросу модель возвращает JSON с intent и slots. Подробнее: https://huggingface.co/datasets/AmazonScience/massive

### Сперва настроим окружение и подготовим код для экспериментов

In [ ]:
# Раскомментируйте в новом окружении, затем перезапустите kernel:
# %pip install "torch==2.8.0" "transformers==4.55.2" "datasets==4.1.1" \
#     "accelerate==1.10.1" "llmcompressor==0.8.1" "compressed-tensors==0.12.2" \
#     "numpy==2.2.6" "pandas==2.2.3" "safetensors>=0.5" "huggingface-hub<1" tqdm

In [ ]:
import gc
import json
import math
import random
import re
import shutil
import struct
from collections import Counter
from collections.abc import Sequence
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from importlib.metadata import version
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoConfig, AutoModelForCausalLM, AutoTokenizer,
    CompressedTensorsConfig, GenerationConfig, PreTrainedModel, PreTrainedTokenizerBase,
    Trainer, TrainingArguments, set_seed,
)

In [ ]:
SEED = 42
ARTIFACTS = Path("artifacts_compression_hw")
ARTIFACTS.mkdir(exist_ok=True)
set_seed(SEED)

def clean_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_empty_dir(path: str | Path) -> Path:
    path = Path(path)
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(f"{path} уже существует и не пустой")
    path.mkdir(parents=True, exist_ok=True)
    return path

environment = {
    p: version(p) for p in (
        "torch", "transformers", "datasets", "llmcompressor",
        "compressed-tensors", "numpy", "safetensors",
    )
}
print(environment)
(ARTIFACTS / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

## Загрузка MASSIVE

Используем датасет MASSIVE. Все калибровочные примеры и данные для дистилляции берём только из train. Оцениваем на валидации (dev).

In [ ]:
DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"  # refs/convert/parquet
LOCALE = "ru-RU"

dataset_url = f"hf://datasets/{DATASET_ID}@{DATASET_REVISION}/{LOCALE}"
data_files = {
    "train": f"{dataset_url}/train/0000.parquet",
    "validation": f"{dataset_url}/validation/0000.parquet",
    "test": f"{dataset_url}/test/0000.parquet",
}
raw = load_dataset("parquet", data_files=data_files)
print(raw)

In [ ]:
example_index = 0
example_raw = raw["train"][example_index]
print("Исходная запись MASSIVE:")
print(json.dumps(
    {key: example_raw[key] for key in ("id", "utt", "annot_utt", "intent")},
    ensure_ascii=False, indent=2,
))

### Сконвертируем данные для калибровки

В MASSIVE текст запроса лежит в поле `utt`, intent записан числом, а slots отмечены прямо в тексте поля `annot_utt`, например `[time : семь утра]`. Модель должна по запросу вернуть JSON с названием intent и списком slots в виде пар «тип — значение». Поэтому преобразуем запись в `text` — исходный запрос — и `gold` — эталонный JSON-ответ. Такой формат удобнее для калибровки и подсчёта метрик.

In [ ]:
SLOT_PATTERN = re.compile(r"\[([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]")

def extract_slots(annot_utt: str) -> list[list[str]]:
    """Достаёт slots из разметки MASSIVE вида `[time : семь утра]`.

    Args:
        annot_utt: размеченный запрос из поля `annot_utt`.

    Returns:
        Список пар `[тип, значение]` в порядке появления в запросе.
    """
    return [[m.group(1).strip(), m.group(2).strip()]
            for m in SLOT_PATTERN.finditer(annot_utt)]

def target_json(intent: str, slots: list[list[str]]) -> str:
    """Собирает эталонный ответ в компактный JSON.

    Args:
        intent: название intent.
        slots: список пар `[тип, значение]`.

    Returns:
        Строка `{"intent":...,"slots":[...]}` без пробелов между полями.
    """
    return json.dumps(
        {"intent": intent, "slots": slots},
        ensure_ascii=False, separators=(",", ":"),
    )

def prepare_massive(raw_splits: DatasetDict) -> DatasetDict:
    """Переводит сплиты MASSIVE в формат запрос - эталонный JSON.

    Args:
        raw_splits: исходные сплиты MASSIVE с полями `id`, `utt`, `intent`, `annot_utt`.

    Returns:
        Сплиты с полями `id`, `text` (запрос) и `gold` (эталонный JSON).
    """
    intent_feature = raw_splits["train"].features["intent"]

    def convert(row: dict[str, Any]) -> dict[str, str]:
        intent = intent_feature.int2str(row["intent"])
        slots = extract_slots(row["annot_utt"])
        return {
            "id": str(row["id"]), "text": row["utt"],
            "gold": target_json(intent, slots),
        }

    return DatasetDict({
        split: ds.map(convert, remove_columns=ds.column_names)
        for split, ds in raw_splits.items()
    })

In [ ]:

data = prepare_massive(raw)
INTENTS = list(raw["train"].features["intent"].names)
SLOT_TYPES = sorted({
    kind for row in raw["train"]
    for kind, value in extract_slots(row["annot_utt"])
})
train_data = data["train"]
DEV_SAMPLES = 256
dev_data = data["validation"]
if DEV_SAMPLES is not None:
    dev_data = dev_data.shuffle(seed=SEED).select(range(min(DEV_SAMPLES, len(dev_data))))
print("Число классов:", len(INTENTS), "типов slots:", len(SLOT_TYPES))

In [ ]:
example_prepared = train_data[example_index]
print("Запрос модели (text):")
print(example_prepared["text"])
print("\nЭталонный ответ (gold):")
print(json.dumps(json.loads(example_prepared["gold"]), ensure_ascii=False, indent=2))

### Генерация ответов
Всем моделям даём одинаковую инструкцию и одинаковые описания допустимых классов. Генерация детерминирована: greedy decoding, одинаковый лимит токенов.

In [ ]:
BASE_MODEL_ID = "Qwen/Qwen3-0.6B" # можно пробовать модели того же семейства, но поменьше, чтобы считалось быстрее

# Значения используются как пример, при необходимости можно регулировать
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 128
GENERATE_BATCH_SIZE = 8

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
system_prompt = (
    'Определи intent и извлеки slots. Верни только JSON вида '
    '{"intent":"...","slots":[["тип","значение"],...]}. '
    'Значения slots копируй из запроса. Если slots нет, верни пустой список. '
    "Не добавляй объяснения.\nДопустимые intents: " + ", ".join(INTENTS)
    + "\nДопустимые типы slots: " + ", ".join(SLOT_TYPES)
)

def encode_prompt(tokenizer: PreTrainedTokenizerBase, text: str) -> list[int]:
    """Токенизирует запрос в chat-формате с общим system prompt.

    Args:
        tokenizer: токенизатор модели с chat template.
        text: запрос пользователя.

    Returns:
        Token ids промпта, заканчивающиеся началом ответа ассистента.
    """
    ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": text}],
        tokenize=True, add_generation_prompt=True, enable_thinking=False,
    )
    if len(ids) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Prompt имеет {len(ids)} токенов. Увеличьте лимит, не обрезайте запрос.")
    return ids

def generation_config(tokenizer: PreTrainedTokenizerBase) -> GenerationConfig:
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id
    return GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=pad_token_id,
        use_cache=True,
    )

def trim_answer(token_ids: list[int], eos_token_id: int, pad_token_id: int) -> list[int]:
    """Отрезает паддинг, после генерации

    Returns:
        Токены ответа; eos остаётся последним, если модель его сгенерировала.
    """
    if eos_token_id in token_ids:
        return token_ids[:token_ids.index(eos_token_id) + 1]
    end = len(token_ids)
    while end and token_ids[end - 1] == pad_token_id:
        end -= 1
    return token_ids[:end]

@torch.inference_mode()
def generate_answer_ids(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    prompts: Sequence[list[int]],
) -> list[list[int]]:
    """Жадно генерирует ответы на батч промптов.

    Args:
        model: causal LM.
        tokenizer: токенизатор модели.
        prompts: token ids промптов; паддинг добавляется слева.

    Returns:
        Токены ответа для каждого промпта в том же порядке, без промпта и паддинга.
    """
    config = generation_config(tokenizer)
    width = max(len(ids) for ids in prompts)
    input_ids, attention_mask = [], []

    for ids in prompts:
        pad = width - len(ids)
        input_ids.append([config.pad_token_id] * pad + ids)
        attention_mask.append([0] * pad + [1] * len(ids))
    device = model.get_input_embeddings().weight.device
    input_ids = torch.tensor(input_ids, device=device)
    attention_mask = torch.tensor(attention_mask, device=device)

    output = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        generation_config=config, use_model_defaults=False, do_sample=False,
    )
    return [
        trim_answer(row, config.eos_token_id, config.pad_token_id)
        for row in output[:, input_ids.shape[1]:].tolist()
    ]

In [ ]:
BASELINE_PATH = ARTIFACTS / "baseline"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()
ensure_empty_dir(BASELINE_PATH)
baseline_model.save_pretrained(BASELINE_PATH, safe_serialization=True)
tokenizer.save_pretrained(BASELINE_PATH)
print("Исходная модель:", BASE_MODEL_ID)
print("Пример prompt, токенов:", len(encode_prompt(tokenizer, train_data[0]["text"])))

## Метрика качества

$$
Q = \tfrac{1}{2} \operatorname{IntentAccuracy} + \tfrac{1}{2} \operatorname{SlotMicroF1}.
$$

Slots сравниваются как **мультимножество** пар `(тип, значение)`: порядок не влияет, лишние дубликаты дают false positives. Неверный JSON или нарушение схемы считается ошибкой всего ответа.

Дополнительно считаем `frame_exact_match` и `valid_json_rate`. Если во всей выборке нет ни одного gold/predicted slot, slot F1 равен 1 только при валидности всех ответов, иначе 0.

In [ ]:
def normalize_value(value: str) -> str:
    """Приводит значение slot к виду для сравнения."""
    return " ".join(value.casefold().split())

def parse_answer(
    text: str, intents: Sequence[str], slot_types: Sequence[str],
) -> dict[str, Any] | None:
    """Разбирает ответ модели и проверяет формат.

    Args:
        text: сырой ответ модели.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с ключами `intent` и `slots` или None, если ответ невалиден.
    """
    try:
        obj = json.loads(text.strip())
        if not isinstance(obj, dict) or set(obj) != {"intent", "slots"}:
            return None
        if not isinstance(obj["intent"], str) or obj["intent"] not in intents:
            return None
        if not isinstance(obj["slots"], list):
            return None
        for slot in obj["slots"]:
            if not isinstance(slot, list) or len(slot) != 2:
                return None
            kind, value = slot
            if (not isinstance(kind, str) or kind not in slot_types
                    or not isinstance(value, str) or not value.strip()):
                return None
        return obj
    except (ValueError, TypeError):
        return None

def slot_counter(obj: dict[str, Any]) -> Counter:
    """Превращает slots ответа в мультимножество

    Args:
        obj: разобранный ответ с ключом `slots`.

    Returns:
        Counter пар `(тип, нормализованное значение)`.
    """
    return Counter((kind, normalize_value(value)) for kind, value in obj["slots"])

def compute_metrics(
    predictions: Sequence[str],
    golds: Sequence[str],
    intents: Sequence[str],
    slot_types: Sequence[str],
) -> dict[str, float]:
    """Считает метрики качества по ответам модели.

    Args:
        predictions: ответы модели.
        golds: эталонные JSON в том же порядке.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с `quality`, `intent_accuracy`, `slot_micro_f1`,
        `frame_exact_match`, `valid_json_rate` и числом примеров `n`.
    """
    if len(predictions) != len(golds) or not golds:
        raise ValueError("Нужны непустые списки одинаковой длины.")
    correct_intent = exact = valid = tp = fp = fn = 0
    for text, gold_text in zip(predictions, golds):
        gold = parse_answer(gold_text, intents, slot_types)
        if gold is None:
            raise ValueError("Невалидная gold-разметка")
        pred = parse_answer(text, intents, slot_types)
        gold_slots = slot_counter(gold)
        pred_slots = Counter() if pred is None else slot_counter(pred)
        tp += sum((pred_slots & gold_slots).values())
        fp += sum((pred_slots - gold_slots).values())
        fn += sum((gold_slots - pred_slots).values())
        if pred is not None:
            valid += 1
            intent_ok = pred["intent"] == gold["intent"]
            correct_intent += int(intent_ok)
            exact += int(intent_ok and pred_slots == gold_slots)
    n = len(golds)
    denom = 2 * tp + fp + fn
    slot_f1 = 2 * tp / denom if denom else float(valid == n)
    intent_acc = correct_intent / n
    return {
        "quality": 0.5 * (intent_acc + slot_f1),
        "intent_accuracy": intent_acc,
        "slot_micro_f1": slot_f1,
        "frame_exact_match": exact / n,
        "valid_json_rate": valid / n,
        "n": n,
    }

def evaluate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Генерирует ответы на выборку батчами и считает метрики.

    Args:
        model: оцениваемая модель.
        tokenizer: её токенизатор.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики из compute_metrics и список ответов модели в порядке выборки.
    """
    model.eval()
    texts = ds["text"]
    predictions = []

    for start in range(0, len(texts), GENERATE_BATCH_SIZE):
        prompts = [encode_prompt(tokenizer, text) for text in texts[start:start + GENERATE_BATCH_SIZE]]
        answers = generate_answer_ids(model, tokenizer, prompts)
        predictions.extend(tokenizer.batch_decode(answers, skip_special_tokens=True))

    metrics = compute_metrics(predictions, ds["gold"], INTENTS, SLOT_TYPES)
    return metrics, predictions

def evaluate_checkpoint(
    model_path: str | Path, ds: Dataset, run_compressed=False,
) -> tuple[dict[str, float], list[str]]:
    """Загружает сохранённый checkpoint, оценивает его и выгружает из памяти.

    Args:
        model_path: каталог dense или квантизированного checkpoint.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики и ответы модели, как в evaluate.
    """
    model_path = str(model_path)
    config = AutoConfig.from_pretrained(model_path)
    quantization_kwargs = {}
    if getattr(config, "quantization_config", None):
        # Веса распаковываются в bf16 на 4-битной сетке
        quantization_kwargs["quantization_config"] = CompressedTensorsConfig(
            run_compressed=run_compressed
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        device_map=DEVICE,
        **quantization_kwargs,
    )
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return evaluate(model, tokenizer, ds)
    finally:
        del model
        clean_memory()

Посчитаем метрики бейзлайна.

In [ ]:
baseline_metrics, baseline_predictions = evaluate(baseline_model, tokenizer, dev_data)
print(baseline_metrics)
(ARTIFACTS / "baseline_dev.json").write_text(
    json.dumps(baseline_metrics, indent=2), encoding="utf-8"
)
results = [{
    "run": "baseline", "path": str(BASELINE_PATH),
    "method": "dense", "calibration": None, **baseline_metrics,
}]
del baseline_model
clean_memory()

## Квантизация (5 баллов)

Используя код с семинара выполните следующие задания:
1) На примере одного из методов откалибруйте бейзлайн на MASSIVE train и на WikiText, сравните качество на dev. Есть ли влияние калибровочного датасета на метрику? (1 балл)

2) Поэкспериментируйте с разными конфигурациями методов AWQ, GPTQ, SmoothQuant (3 балла)

    Запустите минимум три конфигурации, пробуйте менять разные факторы:

    - битность
    - число калибровочных примеров;
    - group size
    - symmetric / asymmetric quantization;
    - smoothing_strength
    - Дополнительно (+1 балл) — какие слои оставить в исходной точности

3) Объясните результат — какие параметры наиболее значимы? (1 балл)


In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier
from llmcompressor.utils import metric_logging

def gpu_memory_usage():
    return [
        (used / total, total / 1e9)
        for free, total in (torch.cuda.mem_get_info(index) for index in range(torch.cuda.device_count()))
        for used in (total - free,)
    ]

metric_logging.get_GPU_memory_usage = gpu_memory_usage

calibration_size = 128
massive_calibration = train_data.shuffle(seed=SEED).select(range(calibration_size))
massive_calibration = Dataset.from_dict({
    "input_ids": [encode_prompt(tokenizer, text) for text in massive_calibration["text"]]
})
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
wiki = wiki.filter(lambda row: len(row["text"].strip()) >= 100)
wiki = wiki.shuffle(seed=SEED).select(range(calibration_size))
wiki_calibration = Dataset.from_dict({
    "input_ids": tokenizer(list(wiki["text"]), truncation=True, max_length=MAX_PROMPT_TOKENS)["input_ids"]
})

for name, method, source, calibration in [
    ("awq_massive", "awq", "massive", massive_calibration),
    ("awq_wiki", "awq", "wikitext", wiki_calibration),
    ("gptq_massive", "gptq", "massive", massive_calibration),
    ("smoothquant_massive", "smoothquant", "massive", massive_calibration),
]:
    if method not in globals().get("QUANT_METHODS", {"awq", "gptq", "smoothquant"}):
        continue
    path = ensure_empty_dir(ARTIFACTS / name)
    model = AutoModelForCausalLM.from_pretrained(
        BASELINE_PATH, torch_dtype=torch.bfloat16, device_map=DEVICE,
    ).eval()
    settings = {"targets": "Linear", "ignore": ["lm_head"]}
    if method == "awq":
        recipe = AWQModifier(scheme="W4A16", **settings)
    elif method == "gptq":
        recipe = GPTQModifier(scheme="W4A16", **settings)
    else:
        recipe = [
            SmoothQuantModifier(smoothing_strength=0.5),
            GPTQModifier(scheme="W8A8", **settings),
        ]
    oneshot(model=model, dataset=calibration, recipe=recipe,
            num_calibration_samples=len(calibration), max_seq_length=MAX_PROMPT_TOKENS)
    model.save_pretrained(path, save_compressed=True)
    tokenizer.save_pretrained(path)
    del model
    clean_memory()
    metrics, _ = evaluate_checkpoint(path, dev_data, run_compressed=method == "smoothquant")
    result = {"run": name, "path": str(path), "method": method, "calibration": source, **metrics}
    print(result, flush=True)
    results.append(result)
    (ARTIFACTS / f"{name}_dev.json").write_text(json.dumps(result, indent=2))
display(pd.DataFrame(results))

### Выводы

AWQ на WikiText дал Quality 0,0380, на MASSIVE 0. GPTQ дал 0,0310. Лучший среди сжатых моделей SmoothQuant с 0,0655 при 0,1031 у BF16

## Hard-label дистилляция (5 баллов)

Теперь реализуем hard-label дистилляцию знаний от учителя **Qwen3-32B** в бейзлайн.

### Задания
1) Измерьте качество учителя на dev и сравните с бейзлайном: насколько учитель сильнее? (1 балл)
2) Получите жадные ответы учителя на сэмпле из train. Используйте батч для ускорения при необходимости (1 балл)
3) Реализуйте код для обучения ученика на ответах учителя (1 балл)
4) Проведите эксперименты с обучением, получите модель ученика, отклоняющуюся на dev не более чем на 10% от метрики учителя (2 балла)


### Измерим качество учителя и сравним с бейзлайном (1 балл)

In [ ]:
STUDENT_MODEL_ID = BASE_MODEL_ID
TEACHER_MODEL_ID = "Qwen/Qwen3-32B"

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
if (
    tokenizer.get_vocab() != teacher_tokenizer.get_vocab()
    or tokenizer.chat_template != teacher_tokenizer.chat_template
    or tokenizer.eos_token_id != teacher_tokenizer.eos_token_id
    or tokenizer.pad_token_id != teacher_tokenizer.pad_token_id
):
    raise ValueError("Токенизаторы ученика и учителя различаются.")
del teacher_tokenizer

teacher_cache = Path("results")
cached_teacher = (teacher_cache / "teacher_dev.json").is_file() and (teacher_cache / "teacher_answers/answers.jsonl").is_file()


In [ ]:
if not cached_teacher:
    teacher_placement = {"device_map": "cpu"}
    if torch.cuda.is_available():
        free_bytes, _ = torch.cuda.mem_get_info()
        teacher_placement = {
            "device_map": "auto",
            "max_memory": {0: min(int(free_bytes * 0.85), free_bytes - 4 * 1024**3), "cpu": "72GiB"},
        }
    teacher = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL_ID, torch_dtype=torch.bfloat16,
        **teacher_placement,
    ).eval()
    student_vocab_size = AutoConfig.from_pretrained(STUDENT_MODEL_ID).vocab_size
    if teacher.config.vocab_size != student_vocab_size:
        raise ValueError("Размерности выходных логитов teacher и student различаются.")

In [ ]:
if cached_teacher:
    teacher_metrics = json.loads((teacher_cache / "teacher_dev.json").read_text())
else:
    teacher_metrics, teacher_predictions = evaluate(teacher, tokenizer, dev_data)
print(teacher_metrics)
(ARTIFACTS / "teacher_dev.json").write_text(json.dumps(teacher_metrics, indent=2))
results.append({"run": "teacher", **teacher_metrics})

### Получим ответы учителя на трейне (1 балл)

Не забывайте выгружать модели и очищать память.

In [ ]:
DISTILL_SAMPLES = 384

def collect_teacher_answers(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    rows: Dataset,
    output_path: str | Path,
) -> Path:
    """Генерируем ответы учителя батчами и сохраняем их в answers.jsonl.

    Args:
        model: модель-учитель.
        tokenizer: общий токенизатор учителя и ученика.
        rows: примеры train с полями `id`, `text`, `gold`.
        output_path: пустой каталог для ответов.

    Returns:
        Каталог с answers.jsonl; в каждой строке `id`, `text`, `gold`,
        `teacher_answer`, `prompt_ids`, `answer_ids`.
    """
    output_path = ensure_empty_dir(output_path)
    records = []
    row_list = list(rows)

    for start in range(0, len(row_list), GENERATE_BATCH_SIZE):
        batch = row_list[start:start + GENERATE_BATCH_SIZE]
        prompts = [encode_prompt(tokenizer, row["text"]) for row in batch]
        answers = generate_answer_ids(model, tokenizer, prompts)
        decoded = tokenizer.batch_decode(answers, skip_special_tokens=True)
        for row, prompt, answer, text in zip(batch, prompts, answers, decoded, strict=True):
            records.append({
                **row, "teacher_answer": text, "prompt_ids": prompt, "answer_ids": answer,
            })

    if not records:
        raise ValueError("Учитель не вернул ни одного ответа.")
    (output_path / "answers.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )
    print(f"Сохранено ответов: {len(records)} / {len(row_list)}")
    return output_path

In [ ]:
if cached_teacher:
    TEACHER_ANSWERS = teacher_cache / "teacher_answers"
else:
    distill_offset = globals().get("DISTILL_OFFSET", 0)
    distill_rows = train_data.shuffle(seed=SEED).select(
        range(distill_offset, min(distill_offset + DISTILL_SAMPLES, len(train_data)))
    )
    TEACHER_ANSWERS = collect_teacher_answers(
        teacher, tokenizer, distill_rows, ARTIFACTS / "teacher_answers"
    )
    del teacher
    clean_memory()

### Реализация пайплайна обучения ученика (1 балл)

Полное дообучение Qwen3-8B на A100 80GB проходит на грани по памяти. Для решения можно включать gradient checkpointing, уменьшать batch или брать меньшую модель того же семейства.

In [ ]:
class TeacherAnswerDataset(torch.utils.data.Dataset):
    """Примеры для обучения ученика на ответах учителя."""

    def __init__(self, path: str | Path) -> None:
        """

        Args:
            path: каталог с answers.jsonl из collect_teacher_answers.
        """
        self.rows = [] # Список примеров из файла

        with (Path(path) / "answers.jsonl").open(encoding="utf-8") as file:
            self.rows = [json.loads(line) for line in file if line.strip()]
        self.rows = [row for row in self.rows if row["answer_ids"]]
        if not self.rows:
            raise ValueError("Нет ответов для обучения")

    def __len__(self) -> int:
        """
            Число строк в answers.jsonl.
        """
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        """
        Args:
            index: номер примера.

        Returns:
            Словарь с `input_ids` (промпт + ответ), `attention_mask` и `labels`,
            токены промпта маскируем значением -100.
        """
        row = self.rows[index]
        input_ids = row["prompt_ids"] + row["answer_ids"]
        labels = [-100] * len(row["prompt_ids"]) + row["answer_ids"]
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

def collate_teacher_answers(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """Паддим примеры справа до общей длины батча.

    Args:
        features: примеры из TeacherAnswerDataset.

    Returns:
        Тензоры `input_ids`, `attention_mask` и `labels` формы [batch, length];
        паддинг в `labels` равен -100.
    """
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    width = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad = width - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad)
        batch["labels"].append(feature["labels"] + [-100] * pad)
    return {key: torch.tensor(value) for key, value in batch.items()}

In [ ]:
def fit_student(name, lr, epochs):
    set_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL_ID, torch_dtype=torch.float32)
    model.config.use_cache = False
    args = TrainingArguments(
        output_dir=str(ARTIFACTS / f"{name}_trainer"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=lr,
        num_train_epochs=epochs,
        bf16=True,
        optim="adafactor",
        gradient_checkpointing=True,
        save_strategy="no",
        logging_steps=10,
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=TeacherAnswerDataset(TEACHER_ANSWERS),
        data_collator=collate_teacher_answers,
    )
    trainer.train()
    path = ensure_empty_dir(ARTIFACTS / name)
    trainer.save_model(path)
    tokenizer.save_pretrained(path)
    del trainer, model
    clean_memory()
    metrics, _ = evaluate_checkpoint(path, dev_data)
    result = {"run": name, "learning_rate": lr, "epochs": epochs, **metrics}
    (ARTIFACTS / f"{name}_dev.json").write_text(json.dumps(result, indent=2))
    results.append(result)
    return result

In [ ]:
student_recipes = globals().get("STUDENT_RECIPES", [(2e-5, 3), (1e-5, 3)])
sft_metrics = fit_student("student_sft", *student_recipes[0])

### Добейтесь максимального качества ученика. Цель — не более 10% отклонения от учителя (2 балла)
Пробуйте менять гиперпараметры, число сэмплов, модели ученика (при условии размера модели ученика не более 8B).

In [ ]:
second_metrics = fit_student("student_sft_second", *student_recipes[1])
if second_metrics["quality"] > sft_metrics["quality"]:
    (ARTIFACTS / "student_sft").rename(ARTIFACTS / "student_sft_first")
    (ARTIFACTS / "student_sft_second").rename(ARTIFACTS / "student_sft")
    (ARTIFACTS / "student_sft_dev.json").rename(ARTIFACTS / "student_sft_first_dev.json")
    second_metrics["run"] = "student_sft"
    sft_metrics["run"] = "student_sft_first"
    (ARTIFACTS / "student_sft_second_dev.json").unlink()
    (ARTIFACTS / "student_sft_first_dev.json").write_text(json.dumps(sft_metrics, indent=2))
    (ARTIFACTS / "student_sft_dev.json").write_text(json.dumps(second_metrics, indent=2))
if "teacher_metrics" in globals():
    print("Best student/teacher:", max(sft_metrics["quality"], second_metrics["quality"]) / teacher_metrics["quality"])
display(pd.DataFrame(results))
pd.DataFrame(results).to_csv(ARTIFACTS / "results.csv", index=False)

### Выводы

За 3 эпохи на 384 ответах учителя с LR 2e-5 Quality студента вырос с 0,1072 до 0,5352. Это 91,7% результата учителя при цели 90%